# Get data for model training

In [90]:
import pandas as pd
from pathlib import Path
import cv2
import sys
sys.path.append("../src/")
import json
from helper_functions.image_functions import show_image

In [ ]:
with open("/home/mellanie/Desktop/Tristan_2026_code/detectron2/model2_50%_col0_data/train.json") as f: train = json.load(f)
# with open("/home/mellanie/Desktop/Tristan_2026_code/detectron2/train.json") as f: test = json.load(f)

train_names = {img["file_name"] for img in train["images"]}
# test_names = {img["file_name"] for img in test["images"]}

# print(f"Overlap: {len(train_names & test_names)}")

print(len(train_names))
# print(len(test_names))



61


In [92]:
SPLIT = 2

# None = drop the shape entirely, otherwise rename to root_hair
LABEL_MAP = {
    "root": None,
    "root_hair": "root_hair",
    "bump": "root_hair",
    "background_root_hair": "root_hair",
    "tangled_root_hair": "root_hair",
    "edge_root_hair": "root_hair",
    "embedded_root_hair": "root_hair",
    "bubble": None,
    "dirt": None,
}

CATEGORIES = [{"id": 0, "name": "root_hair"}]
CATEGORY_NAME_TO_ID = {c["name"]: c["id"] for c in CATEGORIES}

In [ ]:

def get_pairings(filtered):

    images = []
    # jsons = []

    path_stub = "/home/mellanie/Desktop/Annotated_dataset/"
    index = 0
    for _, seed_log in filtered.iterrows():
        date = int(seed_log["Date"])
        nitrogen = seed_log["Nitrate concentration"]
        plate = int(seed_log["Plate"])
        number = int(seed_log["Image number\non the plate "])
        
        name = path_stub + f"{date}_{nitrogen}_Plate{plate}_DAG6_1000_L/{number:06d}.tif"
        label= path_stub + f"{date}_{nitrogen}_Plate{plate}_DAG6_1000_L/labelme_annotations/{number:06d}.json"

        # if seed_log["Ecotype"] == "Col-0*":
        #     if name not in train_names:
        #         continue


        images.append((name, label, seed_log["Ecotype"]))

        index += 1

        

    return images

In [94]:
def labelme_to_coco_single_class(pairs, label_map, category_name_to_id, out_path):
    skipped = 0

    coco = {
        "images": [],
        "annotations": [],
        "categories": [{"id": v, "name": k} for k, v in category_name_to_id.items()],
    }

    ann_id = 0
    for img_id, (img_path, json_path, ecotype) in enumerate(pairs):

        with open(json_path) as f:
            lm = json.load(f)


        coco["images"].append({
            "id": img_id,
            "file_name": img_path,
            "width": lm["imageWidth"],
            "height": lm["imageHeight"],
            "ecotype": ecotype
        })

        for shape in lm["shapes"]:
            orig_label = shape["label"]
            
            if orig_label not in label_map:

                if orig_label == "empty":
                    new_label = None

                elif orig_label in ["edge__root_hair", "tangled", "tangled__root_hair"]:
                    new_label = "root_hair"

                else:
                    raise ValueError(f"Unmapped label '{orig_label}' in {json_path}")


            else:
                new_label = label_map[orig_label]

            if new_label is None:
                continue

            points = shape["points"]
            xs = [p[0] for p in points]
            ys = [p[1] for p in points]
            x_min, y_min = min(xs), min(ys)
            w, h = max(xs) - x_min, max(ys) - y_min

            segmentation = [[coord for point in points for coord in point]]

            coco["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": category_name_to_id[new_label], # single class
                "old_category": orig_label, # multuclass
                "segmentation": segmentation,
                "bbox": [x_min, y_min, w, h],
                "area": w * h,
                "iscrowd": 0,
            })
            ann_id += 1

    with open(out_path, "w") as f:
        json.dump(coco, f, indent=2)

    print(skipped, "skipped")

In [95]:
dataset_info = "Plating_2026-06-30.csv"
data = pd.read_csv(dataset_info)

In [96]:
data.columns

Index(['Date', 'Nitrate concentration', 'Plate', 'Mode', 'Ecotype',
       'Seed number for the specific concentration', 'Annotation', 'Annotator',
       'Seed number on the plate ', 'Image number\non the plate ', 'Stitched',
       'reason for unusability of images ', 'Comment', 'Split_1', 'Split_2',
       'Split_3'],
      dtype='object')

# Get Training Data

In [97]:
# All of these will be loaded and used for the training data
training = data[data[f"Split_{SPLIT}"] == "training"]
training = training[training["Annotation"] == "yes"]
training.head()

,Date,Nitrate concentration,Plate,Mode,Ecotype,Seed number for the specific concentration,Annotation,Annotator,Seed number on the plate,Image number\non the plate,Stitched,reason for unusability of images,Comment,Split_1,Split_2,Split_3
18,20240918.0,N16,2.0,L,Pu2-23,_001,yes,Roberto Zarazua,1.0,1.0,no_need,NaN,NaN,ignore,training,training
19,20240918.0,N16,2.0,L,Pu2-23,_002,yes,Roberto Zarazua,2.0,2.0,no_need,NaN,NaN,ignore,training,training
20,20240918.0,N16,2.0,L,Pu2-23,_003,yes,Roberto Zarazua,3.0,3.0,no_need,NaN,NaN,ignore,training,training
25,20240918.0,N16,2.0,L,Ws-0,_001,yes,Roberto Zarazua,8.0,8.0,no_need,NaN,NaN,ignore,training,training
26,20240918.0,N16,2.0,L,Ws-0,_002,yes,Roberto Zarazua,9.0,9.0,no_need,NaN,NaN,ignore,training,training


In [98]:
# Look at shape
training.shape

(279, 16)

In [99]:
# see distribution of nitrogen
training["Nitrate concentration"].value_counts()

Nitrate concentration
N16     67
N32     67
N64     54
N128    49
N4      42
Name: count, dtype: int64

In [100]:
GENOTYPES = ["Pu2-23",
             "Shahdara",
             "Ws-0",
             "St-0",
             "Oy-0",
             "Uk-1",
             "Bay-0",
             "Kondara",
             "No-0",
             "Hs-0",
             "Cvi-0",
             "Lip-0",
             "id2-3",
             "pme38-1",
             "Ge-0",
             "Ting-1",
             "Ts-1",
             "Col-0*"]

training = training[training["Ecotype"].isin(GENOTYPES)]

In [101]:
# see distribution of ecotypes
training = training[training["Annotation"] == "yes"]

training["Ecotype"].value_counts()

Ecotype
Col-0*      121
id2-3       118
Shahdara     18
Ws-0         12
Pu2-23       10
Name: count, dtype: int64

In [102]:
training.shape

(279, 16)

Optional Random drop

In [103]:
# genotype = "Col-0*"  

# # Get indices of rows matching that genotype
# genotype_rows = training[training["Ecotype"] == genotype]

# # Randomly select 25% of those rows to drop
# rows_to_drop = genotype_rows.sample(frac=0.50, random_state=42).index

# # Drop them from the full dataframe
# training = training.drop(rows_to_drop)

In [104]:
# training["Ecotype"].value_counts()

In [105]:
# training["Ecotype"].value_counts().plot.pie(autopct='%1.1f%%', ylabel='', figsize=(6, 6))

In [106]:
# col0 = training[training["Ecotype"].isin(["Col-0*"])]

# col0_images = get_pairings(col0)


# # # generate cocos
# labelme_to_coco_single_class(col0_images, LABEL_MAP, CATEGORY_NAME_TO_ID, "col0_of_2.json")

# Get Evaluation Data

this will need to be updated to fit the new genotypes when they come

In [107]:
# Genotypes we have at the moment
GENOTYPES = ["Pu2-23",
             "Shahdara",
             "Ws-0",
             "St-0",
             "Oy-0",
             "Uk-1",
             "Bay-0",
             "Kondara",
             "No-0",
             "Hs-0",
             "Cvi-0",
             "Lip-0",
             "id2-3",
             "pme38-1",
             "Ge-0",
             "Ting-1",
             "Ts-1",
             "Col-0*"]


# GENOTYPES = ["Ws-0", "Shahdara"]

In [108]:
evaluation = data[data[f"Split_{SPLIT}"] == "testing"]

In [109]:
evaluation["Nitrate concentration"].value_counts()

Nitrate concentration
N4    360
Name: count, dtype: int64

In [110]:
evaluation = evaluation[evaluation["Ecotype"].isin(GENOTYPES)]


# for eco in evaluation["Ecotype"].unique():
#     if eco in GENOTYPES:
#         print(eco)
#         continue

#     count = (evaluation["Ecotype"] == eco).sum()
#     if count >= 10:
#         print("ecotype not listed", eco)

evaluation = evaluation[(evaluation["Annotation"] == "yes") | (evaluation["Annotation"] == "yes ")]

# evaluation["Ecotype"].value_counts()

In [111]:
evaluation.shape


(180, 16)

In [112]:
evaluation["Ecotype"].value_counts()

Ecotype
Pu2-23      10
Ws-0        10
Shahdara    10
St-0        10
Uk-1        10
Oy-0        10
Kondara     10
Bay-0       10
Hs-0        10
Cvi-0       10
No-0        10
Ts-1        10
Lip-0       10
Ting-1      10
Ge-0        10
id2-3       10
pme38-1     10
Col-0*      10
Name: count, dtype: int64

In [113]:
# evaluation_images = get_pairings(evaluation)
# evaluation_images

# Get data into json

In [ ]:
# Pair images with jsons
training_images = get_pairings(training, train_names)
evaluation_images = get_pairings(evaluation)


# generate cocos
labelme_to_coco_single_class(training_images, LABEL_MAP, CATEGORY_NAME_TO_ID, "train.json")
labelme_to_coco_single_class(evaluation_images, LABEL_MAP, CATEGORY_NAME_TO_ID, "test.json")

0 skipped


In [115]:
# evaluation_images